# ColGraphRAG MultiModalQA (MMQA) — step-by-step pipeline tutorial

This notebook walks through the **query-driven multimodal GraphRAG** pipeline on the **MultiModalQA** corpus (the MMQA counterpart to the WebQA tutorial).

| Step | Script | Role |
|-------|--------|------|
| 0 | `export_mmqa_slice.py` | Normalize and copy `MMQA_*_n*.jsonl` from `data/multimodalqa/dataset/` into `result/<RUN_ID>/mmqa_slice/` |
| 2 | `pattern.py` | Question-specific graph patterns (LLM) |
| 3 | `extraction.py` | Entity and relation extraction (LLM) |
| 4 | `construct.py` | NetworkX → GraphML |
| 5 | `inference.py` | ColEmbed MaxSim retrieval + Gemma answers (`MMGRAPHRAG_DATASET=mmqa`) |
| 6 | `eval/evaluate_multimodal_qa.py` | List EM/F1 and related metrics (MMQA can stratify via `metadata.modalities`) |

**Prerequisites:** Same as the WebQA tutorial — **venv + `pip install -r requirements.txt`**, then **`util/download_models.py`** for Hugging Face weights. Real Gemma inference needs a CUDA GPU. For wiring checks only, enable `--dry-run` in the end-to-end cell.

**Data:** The bundled **dev toy bundle** (e.g. `MMQA_*_n100.jsonl`) should live under `config/data.yaml` → `multimodalqa.data_dir`. Image files are usually under `data/multimodalqa/final_dataset_images/` (filenames from the `path` field in `MMQA_images*.jsonl`).

**Run ID:** MMQA uses `MMGRAPHRAG_RUN_ID=multimodalqa/...`. Outputs collect under `result/multimodalqa/<subdir>/`.

**Two modes:** **End-to-end** (`tests/test_pipeline.py --dataset mmqa`) or **manual step-by-step**. Do not mix different `RUN_ID`s in one session unless you know what you are doing.


## Virtual environment and CUDA

Same as WebQA. Create a venv at the repo root and install `requirements.txt`. Example Jupyter kernel:

```bash
pip install ipykernel
python -m ipykernel install --user --name colgraphrag-mmqa --display-name "Python (colgraphrag_mmqa)"
```

For CUDA PyTorch and the full stack, follow the **Virtual environment** section of `colgraphrag_webqa_pipeline_tutorial_eng.ipynb` (or the Korean MMQA notebook refers to the same steps).

---


In [ ]:
import shutil
import subprocess

nv = shutil.which("nvidia-smi")
if nv:
    r = subprocess.run(
        [
            nv,
            "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    if r.returncode == 0:
        print("nvidia-smi (GPUs):\n")
        print(r.stdout.strip() or "(empty)")
    else:
        print("nvidia-smi failed:", r.stderr)
else:
    print("nvidia-smi not on PATH — no NVIDIA CLI or not a GPU machine.")

print()

try:
    import torch
except ImportError:
    print("torch: not installed — run pip install -r requirements.txt first.")
else:
    print("torch:", torch.__version__)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("torch.cuda.device_count():", n)
        for i in range(n):
            print(f"  [{i}]", torch.cuda.get_device_name(i))
            print("      allocated memory MB:", torch.cuda.memory_allocated(i) / 1e6)


## Resolve repo root

The pipeline needs the repository root on `PYTHONPATH`.


In [ ]:
import os
import sys
from pathlib import Path

_here = Path.cwd()
REPO = None
for _p in [_here, *list(_here.resolve().parents)]:
    if (_p / "inference.py").is_file():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError(
        "Could not find repo root (folder containing inference.py). "
        "cd to the project root or set Jupyter cwd, then re-run this cell."
    )

assert (REPO / "inference.py").is_file(), f"Point REPO at the repo root; got {REPO}"

os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("REPO =", REPO.resolve())
print("cwd =", Path.cwd())


## Environment check (optional)

For real inference, `GEMMA4_E4B_IT_TORCH_DTYPE=bf16` helps reduce VRAM use.


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


## Config files

- `config/data.yaml` — `multimodalqa.data_dir`, `multimodalqa.images_dir`, etc.
- `config/model.yaml` — Gemma and ColEmbed paths.

Put `HF_TOKEN` in `.env` when needed.


In [ ]:
from pathlib import Path

_cfgs = [REPO / "config/data.yaml", REPO / "config/model.yaml"]
_ok = 0
for p in _cfgs:
    ex = p.is_file()
    if ex:
        _ok += 1
    print(p.name, "found:" if ex else "missing:", p)
print("config files ready:", f"{_ok}/{len(_cfgs)}")


## Hugging Face model download

Same as WebQA: run `util/download_models.py` to pull Gemma and ColEmbed into `models/`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "util" / "download_models.py").is_file() else _here.parent

dl = _r / "util" / "download_models.py"
if not dl.is_file():
    raise FileNotFoundError(f"Missing {dl}")

print("=== Download Hugging Face checkpoints (running) ===")
print("Script: util/download_models.py")
print("Targets: gemma, colembed")
print("Gated models may need HF_TOKEN in `.env`.\n")

_dl_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_rc_dl = subprocess.run(
    [sys.executable, "-u", str(dl), "--only", "gemma", "colembed"],
    cwd=str(_r),
    env=_dl_env,
    check=False,
)
print("\nDownload process exit code:", _rc_dl.returncode)


## Simple text QA

Run the next cell to smoke-test Gemma load and generation.


In [ ]:
# Prerequisite: run the "Resolve repo root" cell so REPO exists.
import logging
import os
import sys
import time
from pathlib import Path

if "REPO" not in globals():
    raise RuntimeError("Run the repo root cell first.")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _setup_verbose_logging() -> None:
    fmt = logging.Formatter("%(levelname)s [%(name)s] %(message)s")
    root = logging.getLogger()
    if not any(
        isinstance(h, logging.StreamHandler) and getattr(h, "stream", None) is sys.stdout
        for h in root.handlers
    ):
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(fmt)
        root.addHandler(h)
    root.setLevel(logging.INFO)
    for name in ("mllm.hf_gemma_4_e4b_it", "transformers", "transformers.modeling_utils"):
        logging.getLogger(name).setLevel(logging.INFO)
    try:
        from transformers import logging as tf_logging

        tf_logging.set_verbosity_info()
    except Exception:
        pass


_setup_verbose_logging()

import torch
from mllm.hf_gemma_4_e4b_it import configured, generate_text
from util.llm_defaults import effective_gemma4_e4b_it_model_path

_gemma_dir = effective_gemma4_e4b_it_model_path()
print("Gemma weights:", _gemma_dir)
print("torch / CUDA:", torch.__version__, "|", torch.cuda.is_available())
print("configured:", configured())

if not configured():
    print("Weights not found. Run the download cell above.")
else:
    if torch.cuda.is_available():
        os.environ.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")
        print("dtype hint:", os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE", "(default)"))

    question = "What is the capital of France? Answer in one short sentence."
    print("\n--- Question ---\n", question)
    print("\n--- Generating ---\n")
    t0 = time.perf_counter()
    answer = generate_text(question, max_new_tokens=128)
    elapsed = time.perf_counter() - t0
    print("\n--- Answer ---\n", answer)
    print(f"\n--- Done (wall time {elapsed:.2f}s) ---")


## MMQA toy bundle (dev n100)

If `MMQA_dev_n100.jsonl`, `MMQA_texts_n100.jsonl`, `MMQA_images_n100.jsonl`, and `MMQA_tables_n100.jsonl` exist under `data/multimodalqa/dataset/`, export will work. Otherwise add matching JSONL from the upstream MultiModalQA release or follow the README data-prep steps.


In [ ]:
import json
from collections import Counter
from pathlib import Path

MMQA_DATA = REPO / "data" / "multimodalqa" / "dataset"
print("=== MMQA source data (dataset/) ===")
print("path:", MMQA_DATA.resolve())

for prefix in ("MMQA_dev", "MMQA_texts", "MMQA_images", "MMQA_tables"):
    hits = sorted(MMQA_DATA.glob(prefix + "_n*.jsonl"), reverse=True)
    if hits:
        p = hits[0]
        n = sum(1 for _ in p.open(encoding="utf-8"))
        print(f"  {p.name}: {n} lines")
    else:
        print(f"  (no {prefix}_n*.jsonl)")

q_candidates = sorted(MMQA_DATA.glob("MMQA_dev_n*.jsonl"), reverse=True)
if q_candidates:
    qp = q_candidates[0]
    print("\nQuestion file:", qp.name)
    types = Counter()
    mods = Counter()
    n_rows = 0
    n_chars = 0
    for line in qp.open(encoding="utf-8"):
        line = line.strip()
        if not line:
            continue
        o = json.loads(line)
        n_rows += 1
        n_chars += len(o.get("question") or "")
        meta = o.get("metadata") or {}
        types[str(meta.get("type") or "?")] += 1
        for m in meta.get("modalities") or []:
            mods[str(m)] += 1
    print("  records:", n_rows)
    if n_rows:
        print("  avg question length (chars):", round(n_chars / n_rows, 1))
    print("  metadata.type (top):", dict(types.most_common(8)))
    print("  modalities counts:", dict(sorted(mods.items())))
else:
    print("\nNo MMQA_dev_n*.jsonl — export_mmqa_slice.py will fail.")


### MMQA image preview (optional)

Join `path` from `MMQA_images_n*.jsonl` with `data/multimodalqa/final_dataset_images/`.


In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display

MMQA_DATA = REPO / "data" / "multimodalqa" / "dataset"
IMGS_ROOT = REPO / "data" / "multimodalqa" / "final_dataset_images"

img_files = sorted(MMQA_DATA.glob("MMQA_images_n*.jsonl"), reverse=True)
if not img_files:
    print("No MMQA_images_n*.jsonl under", MMQA_DATA)
elif not IMGS_ROOT.is_dir():
    print("Image root missing:", IMGS_ROOT)
else:
    path = img_files[0]
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print("index:", path.name, "| rows:", len(rows), "| images:", IMGS_ROOT)
    for obj in rows[:5]:
        rel = (obj.get("path") or "").strip()
        p = IMGS_ROOT / rel if rel else None
        if not p or not p.is_file():
            print("skip (missing file):", rel[:80])
            continue
        print(obj.get("id"), (obj.get("title") or "")[:60])
        display(IPyImage(filename=str(p), width=400))


## End-to-end run

`tests/test_pipeline.py --dataset mmqa` runs Phase 0 (export) through evaluation.

- **`-n 5`** — cap on questions (same cap for pattern, extraction, construct, inference).
- **`--dry-run`** — no real LLM/ColEmbed calls (wiring / CPU-friendly).
- For real runs, **remove `--dry-run`** and use a GPU.

Shell equivalent: `bash scripts/mmqa_pipeline_n.sh 5` (real models; sets `MMGRAPHRAG_DATASET=mmqa`, etc.).


In [ ]:
import os
import subprocess
import sys

DRY_RUN = False
N_QUERIES = 2

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
if not DRY_RUN:
    env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

cmd = [
    sys.executable,
    str(REPO / "tests" / "test_pipeline.py"),
    "--dataset",
    "mmqa",
    "-n",
    str(N_QUERIES),
]
if DRY_RUN:
    cmd.append("--dry-run")

print("DRY_RUN=", DRY_RUN, " N_QUERIES=", N_QUERIES)
print("cmd:", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(REPO), env=env)
print("exit code:", r.returncode)

print("\n--- Recent multimodalqa runs ---")
_root = REPO / "result" / "multimodalqa"
if _root.is_dir():
    subs = sorted(_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in subs[:5]:
        pred = (p / "phase5_inference" / "predictions.json").is_file()
        rep = (p / "phase5_inference" / "evaluation_report.json").is_file()
        print(f"  {p.name}  |  predictions={pred}  evaluation_report={rep}")
else:
    print("  result/multimodalqa/ does not exist yet")


## MMQA output layout

After a run you should see under **`result/multimodalqa/<RUN_ID subdir>/`**:

- `mmqa_slice/` — `mmqa_questions.jsonl`, etc.
- `phase2_pattern_cache/`, `phase3_extraction_cache/`
- `phase4_graphs_out/*.graphml`
- `phase5_inference/predictions.json`, `evaluation_report.json`


In [ ]:
import datetime
from pathlib import Path

mmqa_root = REPO / "result" / "multimodalqa"
if mmqa_root.is_dir():
    runs = sorted(mmqa_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in runs[:8]:
        ts = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        pred = (p / "phase5_inference" / "predictions.json").is_file()
        rep = (p / "phase5_inference" / "evaluation_report.json").is_file()
        print(p.name, "| mtime:", ts.isoformat(sep=" ", timespec="seconds"))
        print("    predictions:", pred, "| evaluation_report:", rep)
else:
    print("No result/multimodalqa yet")


## Inspect one GraphML (phase 4)

Same idea as the WebQA tutorial: load the first `result/**/phase4_graphs_out/*.graphml`, print stats, and draw a quick spring layout.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

graphs_dir = REPO / "result"
gml_files = list(graphs_dir.glob("**/phase4_graphs_out/*.graphml"))
if not gml_files:
    print("No GraphML under result/**/phase4_graphs_out/")
else:
    path = sorted(gml_files)[0]
    G = nx.read_graphml(path)
    print("file:", path.relative_to(REPO))
    print("nodes:", G.number_of_nodes(), "  edges:", G.number_of_edges())
    try:
        _Gu = G.to_undirected() if G.is_directed() else G
        print("density (undirected):", f"{nx.density(_Gu):.6f}")
    except Exception as _e:
        print("density skipped:", _e)
    for nid in list(G.nodes)[:3]:
        print(" sample node:", nid, dict(G.nodes[nid]))

    _G = G.to_undirected() if G.is_directed() else G
    _n = _G.number_of_nodes()
    if _n == 0:
        pass
    elif _n > 300:
        print("plot skipped: too many nodes (", _n, ")")
    else:
        fig, ax = plt.subplots(figsize=(7, 5), dpi=110)
        k = 0.6 / max(1, _n ** 0.5)
        pos = nx.spring_layout(_G, seed=0, k=k, iterations=50)
        nx.draw_networkx(
            _G,
            pos=pos,
            ax=ax,
            with_labels=False,
            node_size=180,
            width=0.6,
            edge_color="gray",
            alpha=0.9,
        )
        ax.set_title("GraphML preview — first .graphml under result/")
        ax.axis("off")
        fig.tight_layout()
        plt.show()

## Manual step-by-step

Prefer `RUN_ID` values like `multimodalqa/<experiment_name>` (**include the prefix**). Edit `N_QUERIES` and `DRY_RUN`, then run cells in order.

If you already ran **end-to-end**, use a **new RUN_ID** here or skip this section.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from util.llm_defaults import DEFAULT_GEMMA4_E4B_IT_MODEL_PATH

# --- edit for each experiment ---
RUN_ID = "multimodalqa/notebook_mmqa_manual"
N_QUERIES = 2
DRY_RUN = False

MMQA_DATA_DIR = REPO / "data" / "multimodalqa" / "dataset"
MMQA_IMGS = REPO / "data" / "multimodalqa" / "final_dataset_images"

result_dir = REPO / "result" / RUN_ID
slice_dir = result_dir / "mmqa_slice"
q_file = slice_dir / "mmqa_questions.jsonl"
t_file = slice_dir / "mmqa_texts.jsonl"

dry_run = "1" if DRY_RUN else "0"
gemma_path = os.environ.get("GEMMA4_E4B_IT_MODEL_PATH", "").strip() or str(DEFAULT_GEMMA4_E4B_IT_MODEL_PATH)

common_env = {
    "PYTHONPATH": str(REPO),
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "MMGRAPHRAG_DATASET": "mmqa",
    "WEBQA_RUN_PROFILE": "val_n100",
    "WEBQA_DATA_ROOT": str(REPO / "data" / "webqa"),
    "MMQA_DATA_DIR": str(MMQA_DATA_DIR),
    "PATTERN_MAX_SAMPLES": str(N_QUERIES),
    "WEBQA_EXPORT_MAX": str(N_QUERIES),
    "EXTRACTION_MAX_QUESTIONS": str(N_QUERIES),
    "CONSTRUCT_MAX_QUESTIONS": str(N_QUERIES),
    "INFERENCE_MAX_QUESTIONS": str(N_QUERIES),
    "PATTERN_DRY_RUN": dry_run,
    "EXTRACTION_DRY_RUN": dry_run,
    "PATTERN_CONCURRENCY": "1",
    "EXTRACTION_CONCURRENCY": "1",
    "GEMMA4_E4B_IT_MODEL_PATH": gemma_path,
    "VIDORE_TEXT_LLM_BACKEND": "hf_gemma_4_e4b_it",
    "COLEMBED_MODEL_PATH": os.environ.get(
        "COLEMBED_MODEL_PATH",
        str(REPO / "models" / "retriever" / "llama-nemotron-colembed-vl-3b-v2"),
    ),
}


def run_step(desc: str, cmd: list[str], env: dict) -> int:
    merged = {**os.environ, **env}
    print(f"\n=== {desc} ===\n{' '.join(cmd)}\n")
    r = subprocess.run(cmd, cwd=str(REPO), env=merged)
    print(f"exit code={r.returncode}")
    return r.returncode


if os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE"):
    common_env["GEMMA4_E4B_IT_TORCH_DTYPE"] = os.environ["GEMMA4_E4B_IT_TORCH_DTYPE"]
elif not DRY_RUN:
    common_env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

if DRY_RUN:
    common_env["MMGRAPHRAG_STRICT_REAL"] = "0"

print("RUN_ID:", RUN_ID, "| N_QUERIES:", N_QUERIES, "| DRY_RUN:", DRY_RUN)
print("MMQA_DATA_DIR:", MMQA_DATA_DIR)
print("MMQA images root:", MMQA_IMGS)
print("result_dir:", result_dir)
print("slice_dir (export target):", slice_dir)
print("Gemma:", gemma_path)
print("ColEmbed:", common_env.get("COLEMBED_MODEL_PATH"))


### Phase 0 — `export_mmqa_slice.py`

`MMQA_SLICE_DIR` pins the output directory. You need `MMQA_<split>_n*.jsonl` files for `MMQA_SPLIT` (default `dev`).


In [ ]:
import sys
from pathlib import Path

result_dir.mkdir(parents=True, exist_ok=True)
if slice_dir.exists():
    import shutil

    shutil.rmtree(slice_dir)

export_env = {**common_env, "MMQA_SLICE_DIR": str(slice_dir)}
_rc0 = run_step("Phase 0 MMQA export", [sys.executable, "export_mmqa_slice.py"], export_env)
if _rc0 != 0:
    print("warning: export failed")

_qpath = slice_dir / "mmqa_questions.jsonl"
if _qpath.is_file():
    _nq = sum(1 for _ in _qpath.open(encoding="utf-8"))
    print("slice question lines:", _nq)
else:
    print("missing mmqa_questions.jsonl")


### Pattern (`pattern.py`)

The **pattern** is the **partial-graph skeleton** that aligns with query semantics, as in [Query-Driven Multimodal GraphRAG (Bu et al., ACL 2025 Findings)](https://aclanthology.org/2025.findings-acl.1100/). Here the LLM decides which node and relation types to prioritize per question; the next phase fills that blueprint from evidence.


In [ ]:
pattern_cache = result_dir / "phase2_pattern_cache"
pattern_env = {
    **common_env,
    "PATTERN_JSON_FILE_PATH": str(q_file),
    "PATTERN_CACHE_DIR": str(pattern_cache),
}
_rc_p2 = run_step("Phase 2 pattern", [sys.executable, "pattern.py"], pattern_env)
print("pattern cache:", pattern_cache, "| exists:", pattern_cache.is_dir())


### Inspect the pattern cache

Each `phase2_pattern_cache/*.json` stores the per-question LLM pattern (`response`) and the source question payload (`question`). The next cell lists files and prints a short preview.


In [ ]:
import json

def _pattern_entry_parts(data: dict) -> tuple[str, str, str]:
    q = data.get("question") or {}
    if "qid" in q:
        qid, qtext = q["qid"], q.get("question", "")
    else:
        qid, qtext = q.get("Guid", ""), q.get("Q", "")
    resp = data.get("response", "")
    return str(qid), str(qtext).replace("\n", " "), str(resp).replace("\n", " ")


def _short(s: str, n: int) -> str:
    s = s.strip()
    return s if len(s) <= n else s[: n - 3] + "..."


if not pattern_cache.is_dir():
    print("(no directory)", pattern_cache)
else:
    files = sorted(pattern_cache.glob("*.json"))
    print(f"JSON files: {len(files)}  |  path: {pattern_cache}\n")
    for fp in files:
        with fp.open(encoding="utf-8") as f:
            row = json.load(f)
        qid, qtxt, resp = _pattern_entry_parts(row)
        print("---", fp.name)
        print("  qid:", qid)
        print("  question:", _short(qtxt, 200))
        print("  response:", _short(resp, 280))


### Extraction (`extraction.py`)


In [ ]:
extraction_cache = result_dir / "phase3_extraction_cache"
extraction_env = {
    **common_env,
    "EXTRACTION_QUESTION_FILE": str(q_file),
    "EXTRACTION_TEXT_FILE": str(t_file),
    "EXTRACTION_PATTERN_CACHE_DIR": str(pattern_cache),
    "EXTRACTION_CACHE_DIR": str(extraction_cache),
}
_rc_p3 = run_step("Phase 3 extraction", [sys.executable, "extraction.py"], extraction_env)
print("extraction cache:", extraction_cache, "| exists:", extraction_cache.is_dir())


### Build GraphML (`construct.py`)


In [ ]:
graph_dir = result_dir / "phase4_graphs_out"
construct_env = {
    **common_env,
    "CONSTRUCT_SLICE_DIR": str(slice_dir),
    "CONSTRUCT_QUESTION_FILE": str(q_file),
    "CONSTRUCT_TABLE_FILE": str(slice_dir / "mmqa_tables.jsonl"),
    "CONSTRUCT_IMAGE_FILE": str(slice_dir / "mmqa_images.jsonl"),
    "CONSTRUCT_TEXT_FILE": str(t_file),
    "CONSTRUCT_EXTRACTION_CACHE": str(extraction_cache),
    "CONSTRUCT_OUTPUT_GRAPH_DIR": str(graph_dir),
    "CONSTRUCT_WEBQA_SLICE_DIR": str(slice_dir),
}
_rc_p4 = run_step("Phase 4 graph construction", [sys.executable, "construct.py"], construct_env)
_gml_list = list(graph_dir.glob("*.graphml")) if graph_dir.is_dir() else []
print("GraphML files:", len(_gml_list), "| dir:", graph_dir)


### Inference (`inference.py`)

When `MMGRAPHRAG_DATASET=mmqa`, pass the image root via `MMQA_IMAGES_DIR`.


In [ ]:
import os

phase5_dir = result_dir / "phase5_inference"
phase5_dir.mkdir(parents=True, exist_ok=True)
pred_json = phase5_dir / "predictions.json"

inference_env = {
    **common_env,
    "INFERENCE_GRAPH_DIR": str(graph_dir),
    "INFERENCE_QUESTION_FILE": str(q_file),
    "INFERENCE_OUTPUT_JSON": str(pred_json),
    "INFERENCE_MAX_QUESTIONS": str(N_QUERIES),
    "INFERENCE_DRY_RUN": dry_run,
    "INFERENCE_COLEMBED_RETRIEVAL": os.environ.get("INFERENCE_COLEMBED_RETRIEVAL", "1"),
    "INFERENCE_SLICE_DIR": str(slice_dir),
    "INFERENCE_WEBQA_SLICE_DIR": str(slice_dir),
    "MMQA_IMAGES_DIR": str(MMQA_IMGS if MMQA_IMGS.is_dir() else Path(os.getenv("MMQA_IMAGES_DIR", str(MMQA_IMGS)))),
}
_rc_p5 = run_step("Phase 5 inference", [sys.executable, "inference.py"], inference_env)

if pred_json.is_file():
    import json as _json

    with pred_json.open(encoding="utf-8") as _f:
        _preds = _json.load(_f)
    _keys = list(_preds.keys()) if isinstance(_preds, dict) else []
    print("\npredicted qids:", len(_keys))
    for _qid in _keys[:5]:
        print(f"  qid={_qid}:", str(_preds[_qid])[:240]!r)
else:
    print("no inference output:", pred_json)


### QA evaluation (`eval/evaluate_multimodal_qa.py`)

MMQA uses the same evaluator against gold `mmqa_questions.jsonl`. Adjust `--split_label` for your experiment.


In [ ]:
import json
import sys
from pathlib import Path


def _print_mmqa_report_summary_eng(report_path: Path) -> None:
    def _f(x) -> float:
        try:
            if x is None:
                return 0.0
            return float(x)
        except (TypeError, ValueError):
            return 0.0

    if not report_path.is_file():
        print("evaluation report missing:", report_path)
        return
    with report_path.open(encoding="utf-8") as f:
        rep = json.load(f)
    counts = rep.get("counts") or {}
    print("\n" + "=" * 62)
    print("MMQA QA summary (evaluate_multimodal_qa.py)")
    print("=" * 62)
    print(
        "samples — All:",
        counts.get("All"),
        "| scored qids:",
        counts.get("scored"),
        "| Unimodal:",
        counts.get("Unimodal"),
        "| Multimodal:",
        counts.get("Multimodal"),
    )
    lb = rep.get("leaderboard_summary") or {}
    for bucket in ("All", "Unimodal", "Multimodal"):
        b = lb.get(bucket) or {}
        if not b:
            continue
        print(
            f"  [{bucket}]  QA-FL={_f(b.get('QA-FL')):.4f}  "
            f"QA-Acc={_f(b.get('QA-Acc')):.4f}  QA={_f(b.get('QA')):.4f}"
        )
    scores = rep.get("scores") or {}
    all_s = scores.get("All") or {}
    if all_s:
        print(
            "raw list F1 / list EM:",
            f"{_f(all_s.get('f1')):.4f}",
            "/",
            f"{_f(all_s.get('em')):.4f}",
        )
    print("full JSON:", report_path.resolve())
    print("=" * 62 + "\n")


if pred_json.is_file():
    report_json = phase5_dir / "evaluation_report.json"
    split_label = f"mmqa_dev_n{N_QUERIES}"
    _rc_p6 = run_step(
        "Phase 6 evaluation",
        [
            sys.executable,
            "eval/evaluate_multimodal_qa.py",
            "--predictions",
            str(pred_json),
            "--gold_jsonl",
            str(q_file),
            "--report_json",
            str(report_json),
            "--split_label",
            split_label,
        ],
        {**common_env, "MMGRAPHRAG_RUN_ID": RUN_ID},
    )
    if _rc_p6 == 0:
        _print_mmqa_report_summary_eng(report_json)
    else:
        print("evaluation failed, exit code:", _rc_p6)
else:
    print("skip evaluation: predictions.json missing")


## Appendix: terminal

- **Integrated:** `python tests/test_pipeline.py --dataset mmqa -n 5 [--dry-run]`
- **Shell script:** `bash scripts/mmqa_pipeline_n.sh 5`

For the full env-var wiring, see `tests/test_pipeline.py`.


## Demo UI (optional)

Point `demo/be/config/paths.yaml` at your MMQA `run_id`, slice, and image paths to use the playground like WebQA. See `demo/README.md`.


### Start demo BE + FE from the notebook

Same as the WebQA tutorial: after `REPO` is defined, this backgrounds `demo/be/server.py` and the `demo/fe` Vite dev server.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "demo" / "be" / "server.py").is_file() else _here.parent

be_dir = _repo / "demo" / "be"
fe_dir = _repo / "demo" / "fe"
if not (be_dir / "server.py").is_file():
    raise FileNotFoundError(f"Demo backend not found: {be_dir}")

if "DEMO_PROCESSES" not in globals():
    DEMO_PROCESSES = []
else:
    for _p in list(DEMO_PROCESSES):
        if _p.poll() is None:
            _p.terminate()
    DEMO_PROCESSES.clear()

BE_PORT = int(os.environ.get("DEMO_BE_PORT", "8000"))
FE_PORT = int(os.environ.get("DEMO_FE_PORT", "5173"))

be_env = {**os.environ, "PYTHONPATH": str(be_dir)}
p_be = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", str(BE_PORT)],
    cwd=str(be_dir),
    env=be_env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
DEMO_PROCESSES.append(p_be)
print(f"demo BE started | pid={p_be.pid} | http://127.0.0.1:{BE_PORT}/health")
time.sleep(1.5)

npm = shutil.which("npm")
if npm is None:
    print("npm not on PATH — install Node, then: cd demo/fe && npm install && npm run dev")
else:
    if not (fe_dir / "node_modules").is_dir():
        print("running npm install in demo/fe …")
        subprocess.run([npm, "install"], cwd=str(fe_dir), check=False)
    p_fe = subprocess.Popen(
        [npm, "run", "dev"],
        cwd=str(fe_dir),
        env={**os.environ},
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    DEMO_PROCESSES.append(p_fe)
    print(f"demo FE started | pid={p_fe.pid} | http://127.0.0.1:{FE_PORT}/")

print("\nStop with the next cell.")


### Stop demo servers


In [ ]:
import time as _time

if "DEMO_PROCESSES" in globals() and DEMO_PROCESSES:
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.terminate()
    _time.sleep(0.5)
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.kill()
    DEMO_PROCESSES.clear()
    print("Demo processes stopped.")
else:
    print("No DEMO_PROCESSES to stop.")


## Further reading

- `README.md`
- `notebook/colgraphrag_webqa_pipeline_tutorial_eng.ipynb` — WebQA tutorial (English)
- `notebook/colgraphrag_mmqa_pipeline_tutorial_kor.ipynb` — MMQA tutorial (Korean)
- `tests/test_pipeline.py`, `scripts/mmqa_pipeline_n.sh`
